# 银行客户画像与流失分析

学习项目第 3 个 · 进行中。先复现最初的三个 SQL 客户画像问题，再把这套流程延伸到流失率分析。

已完成部分提供可运行的示例与汇总结果。最后一节保留给后续练习。环境与数据获取方法见 [README](README.md)。

## 1. 数据与统计单位

[Kaggle Bank Customer Churn](https://www.kaggle.com/datasets/radheshyamkollipara/bank-customer-churn) 的 `Customer-Churn-Records.csv`。在此样本中，一行对应一位客户，`CustomerId` 用来检查记录是否重复。

`Geography` 为国家，`Gender` 和 `Age` 是画像字段，`CreditScore`、`Balance` 和 `NumOfProducts` 用于原练习。`Exited` 是下一阶段的流失标签，1 表示已流失，0 表示未流失。

## 2. 读取与检查

从项目根目录启动 Notebook，将 CSV 放在 `data/`。用汇总值检查必要字段的缺失和客户编号重复。编号只用于检查，导入分析表前会去掉。

In [1]:
from contextlib import closing
from pathlib import Path
import sqlite3

import pandas as pd

data_path = Path("data") / "Customer-Churn-Records.csv"
if not data_path.is_file():
    raise FileNotFoundError("找不到 data/Customer-Churn-Records.csv，请按 README 放置数据。")

fields = [
    "CustomerId", "Geography", "Gender", "Age", "CreditScore",
    "Balance", "NumOfProducts", "Exited",
]
customers = pd.read_csv(data_path, encoding="utf-8-sig", usecols=fields)
print("客户记录数：", len(customers))
print("加载的分析字段数：", len(customers.columns))
print("重复客户编号数：", customers["CustomerId"].duplicated().sum())
print("各字段缺失数：")
print(customers.isna().sum().to_string())

analysis = customers.drop(columns="CustomerId")

客户记录数： 10000
加载的分析字段数： 8
重复客户编号数： 0
各字段缺失数：
CustomerId       0
CreditScore      0
Geography        0
Gender           0
Age              0
Balance          0
NumOfProducts    0
Exited           0


## 3. 用 SQLite 复现三个客户画像问题

把分析字段放入内存数据库。以下两类 SQL 写法都用到了：`WHERE` 按条件筛选客户，`GROUP BY` 则按国家分组并计算平均信用分。

In [2]:
age_sql = """
SELECT COUNT(*) AS customers
FROM bank_churn
WHERE Gender = 'Female' AND Age > 40;
"""

credit_sql = """
SELECT Geography, AVG(CreditScore) AS average_credit_score
FROM bank_churn
GROUP BY Geography
ORDER BY Geography;
"""

balance_sql = """
SELECT COUNT(*) AS customers
FROM bank_churn
WHERE Balance > 100000 AND NumOfProducts <= 2;
"""

with closing(sqlite3.connect(":memory:")) as conn:
    analysis.to_sql("bank_churn", conn, index=False)
    age_count = pd.read_sql_query(age_sql, conn).at[0, "customers"]
    credit = pd.read_sql_query(credit_sql, conn)
    balance_count = pd.read_sql_query(balance_sql, conn).at[0, "customers"]

print(f"年龄大于 40 岁的女性客户：{age_count:,}")
print("各国平均信用分：")
print(credit.round(2).to_string(index=False))
print(f"余额超过 100,000 且使用产品不超过 2 个的客户：{balance_count:,}")

年龄大于 40 岁的女性客户：1,704
各国平均信用分：
Geography  average_credit_score
   France                649.67
  Germany                651.45
    Spain                651.33
余额超过 100,000 且使用产品不超过 2 个的客户：4,631


## 4. 用 pandas 核对

用同一张分析表、同一组条件再算一次。均值保留完整精度参与核对，展示时再保留两位小数。

In [3]:
age_filter = (analysis["Gender"] == "Female") & (analysis["Age"] > 40)
balance_filter = (analysis["Balance"] > 100000) & (analysis["NumOfProducts"] <= 2)
assert age_count == age_filter.sum()
assert balance_count == balance_filter.sum()

credit_pandas = analysis.groupby("Geography")["CreditScore"].mean().rename("average_credit_score")
pd.testing.assert_series_equal(credit.set_index("Geography")["average_credit_score"], credit_pandas)
print("三项 SQL 结果与 pandas 一致。")

三项 SQL 结果与 pandas 一致。


### 可以得出什么结论

样本中有 1,704 人符合年龄与性别条件，有 4,631 人符合余额和产品数条件。各国平均信用分相近，只能说明这几个分组的均值相近。信贷风险、是否活跃、是否有理财需求都需要对应的变量和进一步的证据。

## 5. 下一题：各国样本流失率（待完成）

从 `analysis` 开始，沿用第 3 节的内存 SQLite 方式。按 `Geography` 统计每组的**客户总数、流失人数和流失率**，并按流失率降序显示。流失率的分母是该组的客户数。

运行后，用 pandas 核对分组结果，并检查各组人数之和是否等于总样本量。最后用一句话说明排名对应的是当前样本。之后可以继续比较其他分组，再加入图表和模型。

In [4]:
# 在这里写按国家计算流失率的 SQL，再用 pandas 核对。